# 1. Exploratory Data Analysis

Goal: understand the shape, distributions, and relationships in the data before touching it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('../data/raw/train.csv')
print('Shape:', df.shape)
df.head()

## Basic info

In [ ]:
df.info()
df.describe()

## Target variable: SalePrice distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['SalePrice'], kde=True, ax=axes[0], color='#1D9E75')
axes[0].set_title('SalePrice — raw (right-skewed)')

sns.histplot(np.log1p(df['SalePrice']), kde=True, ax=axes[1], color='#534AB7')
axes[1].set_title('log(SalePrice) — more normal')

plt.tight_layout()
plt.savefig('../reports/figures/saleprice_distribution.png', dpi=150)
plt.show()

print(f'Skewness raw:  {df["SalePrice"].skew():.2f}')
print(f'Skewness log:  {np.log1p(df["SalePrice"]).skew():.2f}')

## Missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'{len(missing)} columns have missing values')

plt.figure(figsize=(10, 6))
missing.plot(kind='bar', color='#D85A30')
plt.title('Missing values per column')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150)
plt.show()

## Correlation with SalePrice

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

plt.figure(figsize=(8, 12))
corr.plot(kind='barh', color='#1D9E75')
plt.title('Feature correlation with SalePrice')
plt.xlabel('Pearson r')
plt.tight_layout()
plt.savefig('../reports/figures/correlations.png', dpi=150)
plt.show()

print('Top 10 positively correlated:')
print(corr.head(10))

## Scatter plots: top features vs SalePrice

In [ ]:
top_features = corr.head(5).index.tolist()

fig, axes = plt.subplots(1, len(top_features), figsize=(18, 4))
for ax, feat in zip(axes, top_features):
    ax.scatter(df[feat], df['SalePrice'], alpha=0.3, s=10, color='#378ADD')
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')

plt.suptitle('Top 5 features vs SalePrice', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/scatter_top_features.png', dpi=150)
plt.show()

## Correlation heatmap (top 15 features)

In [ ]:
top15 = corr.head(15).index.tolist() + ['SalePrice']
plt.figure(figsize=(12, 10))
sns.heatmap(df[top15].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation heatmap — top 15 features')
plt.tight_layout()
plt.savefig('../reports/figures/heatmap.png', dpi=150)
plt.show()